## RoBERTa model: fine-tuned for binary sequence classification on the SRAQ dataset

In [1]:
# imports

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import torch
import pandas as pd
from transformers import AutoTokenizer
from datasets import Dataset
from transformers import DataCollatorWithPadding, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import f1_score

/home/gusbeihel@GU.GU.SE/ml2-final-project/.venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# checking pytorch ???

print(torch.cuda.is_available())

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

True
PyTorch: 2.14.1+cu130
CUDA build: 13.0
CUDA available: True
0 NVIDIA A30


In [4]:
# loading processed data

train = pd.read_json("../data/train_processed.jsonl", lines=True)
valid = pd.read_json("../data/dev_processed.jsonl", lines=True)
test = pd.read_json("../data/test_processed.jsonl", lines=True)

### Tokenization

In [5]:
# loading tokenizer

tokenizer = AutoTokenizer.from_pretrained("FacebookAI/roberta-base")

# looking at an example
question = train["question"].iloc[0]
tokens = tokenizer.tokenize(question)
print(tokens)

['Are', 'Ġwe', 'Ġentitled', ',', 'Ġfor', 'Ġexample', ',', 'Ġto', 'Ġrequire', 'Ġthat', 'Ġnew', 'Ġimmigrants', 'Ġbe', 'Ġof', 'Ġa', 'Ġcertain', 'Ġethnic', 'Ġor', 'Ġreligious', 'Ġbackground', ',', 'Ġare', 'Ġwe', 'Ġentitled', 'Ġto', 'Ġrequire', 'Ġthat', 'Ġthey', 'Ġpay', 'Ġextremely', 'Ġhigh', 'Ġfees', 'Ġin', 'Ġreturn', 'Ġfor', 'Ġcitizenship', ',', 'Ġare', 'Ġwe', 'Ġallowed', 'Ġattach', 'Ġconditions', 'Ġwhich', 'Ġhold', 'Ġeven', 'Ġafter', 'Ġcitizenship', 'Ġhas', 'Ġbeen', 'Ġgranted', ',', 'Ġand', 'Ġif', 'Ġnot', ',', 'Ġwhy', 'Ġnot', '?']


In [6]:
# mapping "rhetorical" and "informational" labels to integers

label2id = {"informational": 0, "rhetorical": 1}
id2label = {0: "informational", 1: "rhetorical"}

# creating new column with integers
train["label_int"] = train["label"].map(label2id)

valid["label_int"] = valid["label"].map(label2id)

test["label_int"] = test["label"].map(label2id)

In [7]:
# truncating text that exceeds the maximum length

# truncating from the left (beginning)
tokenizer.truncation_side = "left"

def preprocess(context, question):
    return tokenizer(
        context,
        question,
        truncation="only_first", # truncates only context (from the left)
        max_length=512
    )

I truncated the text from the left to preserve the context closest to the question.

In [20]:
# tokenizing training data

train_tokenized = train.apply(
    lambda row: preprocess(row["preceding_turns"], row["question"]),
    axis=1
)

# checking
print(train_tokenized.iloc[0].keys())
print(len(train_tokenized.iloc[0]["input_ids"]))

# converting tokenized data into dataframe
train_encoded = pd.DataFrame(train_tokenized.tolist())

# adding integer labels
train_encoded["labels"] = train["label_int"].to_numpy()

train_encoded.head()

KeysView({'input_ids': [0, 74, 202, 28, 10, 538, 3855, 6, 25, 171, 3354, 3458, 82, 74, 45, 28, 2460, 7, 185, 215, 41, 14270, 4, 38, 64, 45, 4744, 10, 36906, 3354, 3346, 145, 2882, 7, 342, 39, 865, 15, 5, 42244, 8, 24909, 37, 2046, 11, 383, 14, 32, 11, 15441, 20874, 7, 39812, 488, 4, 152, 74, 1266, 24, 74, 33, 10, 1233, 10142, 913, 11, 4881, 5, 936, 7510, 566, 92, 4175, 6, 150, 45, 145, 10, 7926, 7, 7212, 453, 9, 70, 3458, 34332, 4, 50118, 50118, 24788, 35, 598, 2519, 7, 5, 1049, 8823, 38, 524, 562, 35, 50118, 50118, 1640, 134, 43, 22, 1213, 115, 95, 6105, 111, 99, 18, 5, 477, 845, 1009, 100, 33, 460, 2334, 7, 42, 11, 127, 1461, 618, 4, 286, 1840, 18, 12253, 6, 2073, 5, 16490, 618, 137, 6827, 8, 21025, 961, 1493, 18, 86, 44460, 287, 2801, 6, 150, 171, 74, 28, 2882, 7, 6105, 6, 5, 38004, 7958, 74, 533, 45, 28, 11, 41, 14270, 7, 1840, 4, 10623, 52, 74, 1888, 5, 346, 9, 38004, 12, 25331, 23907, 16281, 6, 8, 33, 5, 2194, 14547, 31, 5, 786, 12, 5982, 1023, 9997, 7958, 16281, 4, 280, 16, 41, 

,input_ids,attention_mask,labels
0,"[0, 74, 202, 28, 10, 538, 3855, 6, 25, 171, 33...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0
1,"[0, 41761, 1215, 250, 35, 20560, 203, 99, 24, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0
2,"[0, 41761, 1215, 250, 35, 178, 141, 171, 82, 7...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1
3,"[0, 41761, 1215, 250, 35, 20, 132, 207, 9061, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1
4,"[0, 41761, 1215, 250, 35, 345, 16, 117, 3050, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0


In [21]:
# converting to Hugging Face dataset

train_dataset = Dataset.from_pandas(
    train_encoded,
    preserve_index=False
)

print(train_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 345
})


In [22]:
# tokenizing validation data

valid_tokenized = valid.apply(
    lambda row: preprocess(row["preceding_turns"], row["question"]),
    axis=1
)

# checking
print(valid_tokenized.iloc[0].keys())
print(len(valid_tokenized.iloc[0]["input_ids"]))

# converting tokenized data into dataframe
valid_encoded = pd.DataFrame(valid_tokenized.tolist())

# adding integer labels
valid_encoded["labels"] = valid["label_int"].to_numpy()

valid_encoded.head()

KeysView({'input_ids': [0, 41761, 1215, 250, 35, 38, 437, 11, 5, 609, 9, 2159, 10, 20568, 8, 65, 9, 127, 934, 1379, 16, 127, 19960, 18, 10029, 4, 1437, 264, 855, 34, 155, 3678, 8, 132, 10017, 4, 1437, 264, 18, 10, 1307, 3477, 16095, 8, 10, 9087, 2903, 98, 79, 14534, 13, 106, 157, 8, 51, 214, 157, 30758, 36, 26837, 5, 6185, 4758, 6, 38, 24909, 37, 18, 5, 22406, 322, 50118, 50118, 11913, 5924, 6, 144, 9, 5, 2127, 38, 437, 546, 7, 907, 34, 10, 155, 4716, 714, 3000, 4, 1437, 38, 437, 45, 3915, 59, 65, 9, 5, 10017, 142, 24, 18, 41, 11894, 4758, 8, 5907, 40, 655, 192, 123, 4, 50118, 50118, 1213, 214, 70, 157, 30758, 8, 218, 75, 28847, 98, 38, 218, 75, 206, 24, 581, 28, 10, 380, 936, 6, 53, 127, 129, 2490, 16, 114, 38, 362, 5, 3678, 15, 5792, 23, 363, 82, 74, 192, 155, 8, 77, 38, 213, 66, 7, 120, 5, 6185, 4758, 23, 363, 51, 581, 216, 38, 33, 23, 513, 204, 4, 1437, 38, 348, 16425, 11, 34003, 137, 8, 216, 89, 32, 460, 6611, 54, 32, 23809, 8, 2882, 7, 11316, 7, 5, 289, 16024, 59, 932, 4, 50118, 

,input_ids,attention_mask,labels
0,"[0, 41761, 1215, 250, 35, 38, 437, 11, 5, 609,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0
1,"[0, 41761, 1215, 250, 35, 646, 5593, 12677, 74...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1
2,"[0, 41761, 1215, 250, 35, 280, 2682, 16, 4968,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1
3,"[0, 41761, 1215, 250, 35, 359, 19377, 131, 100...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1
4,"[0, 41761, 1215, 250, 35, 38, 206, 14, 31682, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1


In [23]:
# converting to Hugging Face dataset

valid_dataset = Dataset.from_pandas(
    valid_encoded,
    preserve_index=False
)

print(valid_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 90
})


In [24]:
# TEST !!!

### Training

In [25]:
# implementing dynamic padding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [26]:
# defining evaluation function to compute macro F1

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "macro_f1": f1_score(labels, predictions, average="macro")
    }

In [ ]:
# training model

model = AutoModelForSequenceClassification.from_pretrained(
    "FacebookAI/roberta-base", num_labels=2, id2label=id2label, label2id=label2id
)

training_args = TrainingArguments(
    output_dir="../model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    load_best_model_at_end=True,
    seed=42,
    data_seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(trainer.args.device)

trainer.train()

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 12309.36it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


cuda:0


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.653401,0.391892
2,No log,0.653259,0.391892
3,No log,0.649935,0.391892


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.07s/it]


TrainOutput(global_step=66, training_loss=0.6759086377692946, metrics={'train_runtime': 49.0767, 'train_samples_per_second': 21.089, 'train_steps_per_second': 1.345, 'total_flos': 272319942297600.0, 'train_loss': 0.6759086377692946, 'epoch': 3.0})

 First training attempt; macro F1 did not improve at all  
 results suggest model is predicting every question as RQ
 
 [66/66 00:50, Epoch 3/3]  
Epoch	Training Loss	Validation Loss	Macro F1  
1	No log	0.653401	0.391892  
2	No log	0.653259	0.391892  
3	No log	0.649935	0.391892  

In [34]:
predictions = trainer.predict(valid_dataset)
y_pred = np.argmax(predictions.predictions, axis=-1)

print(np.unique(y_pred, return_counts=True))

(array([1]), array([90]))


confirmed: all instances predicted as RQs  

- class imbalance
- small dataset
- too few epochs
- long contexts


In [35]:
# trying more epochs

# training model

model = AutoModelForSequenceClassification.from_pretrained(
    "FacebookAI/roberta-base", num_labels=2, id2label=id2label, label2id=label2id
)

training_args = TrainingArguments(
    output_dir="../model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=6,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    load_best_model_at_end=True,
    seed=42,
    data_seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(trainer.args.device)

trainer.train()

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 11169.69it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


cuda:0


Epoch,Training Loss,Validation Loss,Macro F1
1,0.685625,0.651697,0.391892
2,0.689844,0.656316,0.391892
3,0.668673,0.649849,0.391892
4,0.650499,0.649746,0.391892
5,0.638073,0.648717,0.391892
6,0.624852,0.654876,0.391892


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.05s/it]


TrainOutput(global_step=132, training_loss=0.6595943985563336, metrics={'train_runtime': 96.254, 'train_samples_per_second': 21.506, 'train_steps_per_second': 1.371, 'total_flos': 544639884595200.0, 'train_loss': 0.6595943985563336, 'epoch': 6.0})